# Admin Setup — Production-Ready ML Systems Workshop

**Run once before the workshop. Requires ACCOUNT ADMIN or workspace admin.**

This notebook:
1. Creates the shared schema and UC Volume
2. Loads the Telco churn CSV into a managed Delta table
3. Creates a held-out evaluation table
4. Grants all workspace users read access to shared data

> **Pre-requisite for Step 4:** Upload `Telco-Customer-Churn.csv` to the Volume created in Step 3 before running the data load cells.

In [0]:
dbutils.widgets.text("catalog", "workshop", "Catalog Name")
catalog = dbutils.widgets.get("catalog")
print(f"Using catalog: {catalog}")

## Step 1: Set Active Catalog

In [0]:
%sql
USE CATALOG ${catalog}

## Step 2: Create Shared Schema

Prefixed with `00_` so it sorts first alphabetically in Catalog Explorer.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS `00_shared`
  COMMENT 'Shared read-only data for all workshop participants'

## Step 3: Create UC Volumes


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS `00_shared`.raw_files
  COMMENT 'Staging area for raw workshop data files';

CREATE VOLUME IF NOT EXISTS `00_shared`.wheels
  COMMENT 'Staging area for workshop Python wheels';

## Step 3a: Copy the Starter CSV File to the Volume


In [0]:
import shutil

# Derive the bundle root from this notebook's own workspace path.
# Works regardless of which user deployed the bundle or where it was synced.
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# e.g. /Users/dennis.schultz/Production-Ready-ML-workshop/src/utils/01_instructor_setup
bundle_root = "/Workspace" + notebook_path.rsplit("/scripts/", 1)[0]
artifact_src = f"{bundle_root}/artifacts"

shutil.copy(
  f"{artifact_src}/Telco-Customer-Churn.csv",
  f"/Volumes/{catalog}/00_shared/raw_files/Telco-Customer-Churn.csv"
)

## Step 3b: Copy the churn_model Wheel

Copies the uploaded whl file to a volume for reference by notebooks later

In [0]:
shutil.copy(
  f"{artifact_src}/churn_model-0.1.0-py3-none-any.whl",
  f"/Volumes/{catalog}/00_shared/wheels/"
)

## Step 4: Load Telco CSV into Delta Table

> ⚠️ Only run after uploading `Telco-Customer-Churn.csv` to the Volume in Step 3.

Casts all columns to the correct types. Handles the space-padded `TotalCharges` nulls present in the raw IBM data.

In [0]:
%sql
CREATE OR REPLACE TABLE `00_shared`.telco_churn
  COMMENT 'IBM Telco Customer Churn dataset — 7,043 customers, 21 features'
AS
SELECT
  customerID,
  gender,
  CAST(SeniorCitizen AS INT)              AS SeniorCitizen,
  Partner,
  Dependents,
  CAST(tenure AS INT)                     AS tenure,
  PhoneService,
  MultipleLines,
  InternetService,
  OnlineSecurity,
  OnlineBackup,
  DeviceProtection,
  TechSupport,
  StreamingTV,
  StreamingMovies,
  Contract,
  PaperlessBilling,
  PaymentMethod,
  CAST(MonthlyCharges AS DOUBLE)          AS MonthlyCharges,
  CAST(
    CASE WHEN TotalCharges = ' ' THEN NULL ELSE TotalCharges END
    AS DOUBLE
  )                                       AS TotalCharges,
  Churn
FROM read_files(
  '/Volumes/${catalog}/00_shared/raw_files/Telco-Customer-Churn.csv',
  format      => 'csv',
  header      => true,
  inferSchema => false,
  schema      => '
    customerID STRING,
    gender STRING,
    SeniorCitizen STRING,
    Partner STRING,
    Dependents STRING,
    tenure STRING,
    PhoneService STRING,
    MultipleLines STRING,
    InternetService STRING,
    OnlineSecurity STRING,
    OnlineBackup STRING,
    DeviceProtection STRING,
    TechSupport STRING,
    StreamingTV STRING,
    StreamingMovies STRING,
    Contract STRING,
    PaperlessBilling STRING,
    PaymentMethod STRING,
    MonthlyCharges STRING,
    TotalCharges STRING,
    Churn STRING
  '
)

In [0]:
%sql
-- Verify the load — expected: 7,043 rows
SELECT COUNT(*) AS row_count FROM `00_shared`.telco_churn

## Step 5: Create Holdout Table

Stratified 15% sample held out from all training exercises.
Used in Session 2 for realistic model evaluation against unseen data.

In [0]:
%sql
CREATE OR REPLACE TABLE `00_shared`.telco_churn_holdout
  COMMENT 'Held-out 15% sample — not used during any training exercise'
AS
SELECT * FROM `00_shared`.telco_churn
WHERE customerID IN (
  SELECT customerID FROM (
    SELECT customerID,
           ROW_NUMBER() OVER (PARTITION BY Churn ORDER BY RAND(42)) AS rn,
           COUNT(*) OVER (PARTITION BY Churn) AS cnt
    FROM `00_shared`.telco_churn
  )
  WHERE rn <= CEIL(cnt * 0.15)
)

In [0]:
%sql
-- Verify the holdout — expected: ~1,056 rows
SELECT COUNT(*) AS holdout_count FROM `00_shared`.telco_churn_holdout

## Step 6: Grant Permissions to All Workspace Users

In [0]:
%sql
GRANT USE CATALOG ON CATALOG ${catalog} TO `account users`;
GRANT CREATE SCHEMA ON CATALOG ${catalog} TO `account users`;
GRANT BROWSE ON CATALOG ${catalog} TO `account users`;
GRANT USE SCHEMA ON SCHEMA ${catalog}.`00_shared` TO `account users`;
GRANT SELECT ON TABLE ${catalog}.`00_shared`.telco_churn TO `account users`;
GRANT SELECT ON TABLE ${catalog}.`00_shared`.telco_churn_holdout TO `account users`;
GRANT READ VOLUME ON VOLUME ${catalog}.`00_shared`.raw_files TO `account users`;
GRANT READ VOLUME ON VOLUME ${catalog}.`00_shared`.wheels TO `account users`;

## Step 7: Create the Foundation Model endpoint


In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    AiGatewayConfig,
    AiGatewayGuardrailParameters,
    AiGatewayGuardrailPiiBehavior,
    AiGatewayGuardrailPiiBehaviorBehavior,
    AiGatewayGuardrails,
    AiGatewayInferenceTableConfig,
    AiGatewayRateLimit,
    AiGatewayRateLimitKey,
    AiGatewayRateLimitRenewalPeriod,
    AiGatewayUsageTrackingConfig,
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()
endpoint_name = f"foundation_model_with_gateway"
foundation_model = "system.ai.llama_v3_3_70b_instruct"

# -- 1. Delete the existing endpoint (if any) --
try:
    w.serving_endpoints.delete(name=endpoint_name)
    print(f"\u2713 Deleted existing endpoint '{endpoint_name}'")
    time.sleep(10)
except Exception:
    pass

# -- 2. Create the endpoint with the UC foundation model --
print(f"Creating endpoint: {endpoint_name}")
print("This will take a few minutes...")

w.serving_endpoints.create_and_wait(
    name=endpoint_name,
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=foundation_model,
                entity_version="1",
                min_provisioned_throughput=0,
                max_provisioned_throughput=600,
                scale_to_zero_enabled=True,
            )
        ]
    ),
    ai_gateway=AiGatewayConfig(
        usage_tracking_config=AiGatewayUsageTrackingConfig(enabled=True),
        inference_table_config=AiGatewayInferenceTableConfig(
            catalog_name=catalog,
            schema_name="00_shared",
            table_name_prefix="fm_gateway",
            enabled=True,
        ),
        guardrails=AiGatewayGuardrails(
            input=AiGatewayGuardrailParameters(
                safety=True,
                pii=AiGatewayGuardrailPiiBehavior(
                    behavior=AiGatewayGuardrailPiiBehaviorBehavior.BLOCK,
                ),
            ),
            output=AiGatewayGuardrailParameters(
                safety=True,
                pii=AiGatewayGuardrailPiiBehavior(
                    behavior=AiGatewayGuardrailPiiBehaviorBehavior.BLOCK,
                ),
            ),
        ),
        rate_limits=[
            AiGatewayRateLimit(
                calls=60,
                key=AiGatewayRateLimitKey.USER,
                renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
            )
        ],
    ),
)

print(f"\n\u2713 Endpoint '{endpoint_name}' is Ready!")
print(f"  Model          : {foundation_model}")
print(f"  Inference table: {catalog}.00_shared.fm_gateway_payload")
print(f"  Rate limit     : 60 requests/min per user")
print(f"  Guardrails     : Safety + PII blocking (input & output)")
print(f"  Usage tracking : Enabled")

## Model endpoint permissions
GRANT CAN MANAGE ON the endpoint to workshop users

## Final Verification

In [0]:
%sql
SELECT
  'Setup complete'                                               AS status,
  (SELECT COUNT(*) FROM `00_shared`.telco_churn)         AS training_rows,
  (SELECT COUNT(*) FROM `00_shared`.telco_churn_holdout) AS holdout_rows